# Symmetry Phase 2 - Theory Diagnostics

This notebook shows how to:
1. diagnose translation symmetry,
2. fit an exponential kernel,
3. compare empirical and fitted PCA geometry.

In [ ]:
import numpy as np

from boardGPT.analysis import (
    PolicyConfig,
    SyntheticWorld,
    build_cooccurrence_statistics,
    build_exponential_kernel_matrix,
    fit_exponential_kernel_from_matrix,
    generate_synthetic_sequences,
    lattice_1d_concept,
    pairwise_distances,
    pca_from_gram,
    translation_symmetry_diagnostic,
)

## 1) Build a periodic 1D synthetic world

In [ ]:
concept = lattice_1d_concept(num_tokens=48, periodic=True, period=2.0, name='ring1d')
world = SyntheticWorld(concept=concept)
dist = pairwise_distances(concept)

cfg = PolicyConfig(
    length=120,
    num_sequences=3000,
    seed=123,
    transition_kernel='exponential',
    transition_scale=0.18,
    transition_strength=1.0,
    epsilon_uniform=0.02,
)
ds = generate_synthetic_sequences(world, cfg)
stats = build_cooccurrence_statistics(ds.sequences, vocab_size=world.vocab_size, window_size=8)
matrix = stats.m_star
print('matrix shape:', matrix.shape)

## 2) Translation-symmetry score

In [ ]:
diag = translation_symmetry_diagnostic(matrix=matrix, distances=dist, num_bins=24)
print('symmetry R^2:', diag.r2)
print('symmetry MSE:', diag.mse)

## 3) Fit exponential kernel

In [ ]:
fit = fit_exponential_kernel_from_matrix(matrix=matrix, distances=dist, num_bins=24, fit_offset=True)
print(fit)

## 4) Compare empirical vs fitted Gram geometry

In [ ]:
gram_fit = build_exponential_kernel_matrix(
    distances=dist,
    amplitude=fit.amplitude,
    sigma=fit.sigma,
    offset=fit.offset,
)

coords_emp, evals_emp, _ = pca_from_gram(matrix, rank=3, center=True)
coords_fit, evals_fit, _ = pca_from_gram(gram_fit, rank=3, center=True)

print('Top evals empirical:', evals_emp[:3])
print('Top evals fitted   :', evals_fit[:3])
print('Empirical first point (PC1,PC2):', coords_emp[0, :2])
print('Fitted first point    (PC1,PC2):', coords_fit[0, :2])